### Embeddings API

In [487]:
import json
import requests

url = "https://bge-m3.run.fc.uralsibbank.ru/v1/embeddings"

headers = {
    "Content-Type": "application/json"
}

payload = {
    "input": "What is Deep Learning?",
    "model": "BAAI/bge-m3",
    "encoding_format": "float"
}

response = requests.post(url, headers=headers, data=json.dumps(payload))

# Вывод статуса и тела ответа
print("Status code:", response.status_code)
# print("Response body:", response.text)

Status code: 200


### LLM API

In [314]:
import requests
import json

url = "https://gpt.run.fc.uralsibbank.ru/v1/chat/completions"
headers = {"Content-Type": "application/json"}
payload = {
    "model": "openai/gpt-oss-120b",
    "messages": [
        {"role": "user", "content": pr}
    ],
    "temperature": 0.5,
    "top_p": 0.95,
    "top_k": 20,
    "max_tokens": 2048,
    "response_format": {"type": "json_object"}
}

response = requests.post(url, headers=headers, data=json.dumps(payload))
# print(response.status_code)
# print(response.text)

In [315]:
# a = json.loads(response.content)['choices'][0]['message']['content']
# print(a)

### Гоним запросы через ллм

In [344]:
sp = list()
for i, row in tqdm.tqdm(df.iterrows()):
    with open('prompt.txt', 'r') as f:
        pr = f.read()
    pr = pr.replace('{VARIABLE}', row['desc'])
    payload['messages'][0]['content'] = pr
    response = requests.post(url, headers=headers, data=json.dumps(payload))
    a = json.loads(response.content)['choices'][0]['message']['content']
    sp.append(a)
    # print(a)
    if i==1000:
        break
    
    

1000it [36:45,  2.21s/it]


In [7]:
40*25

1000

# Обработка

In [345]:
len(sp)

1001

In [349]:
df_small = df.iloc[:1001]

In [350]:
df_small = df_small.reset_index(drop=True)

In [351]:
df_small['sp'] = sp

In [352]:
q = pd.DataFrame().from_dict(df_small['sp'].apply(lambda a: json.loads(a)).tolist())

In [356]:
q['is'] = q.issues.apply(lambda x: get_values(x, 'type') if isinstance(x, list) else x)
q['is_d'] = q.issues.apply(lambda x: get_values(x, 'description') if isinstance(x, list) else x)
q['ref'] = q.requested_actions.apply(lambda x: get_values(x, 'type') if isinstance(x, list) else x)
q['ref_d'] = q.requested_actions.apply(lambda x: get_values(x, 'description') if isinstance(x, list) else x)

In [357]:
def get_values(dicts, key):
    if isinstance(dicts, list):
        return [d.get(key) for d in dicts]

In [358]:
q['is'].explode().value_counts()

Неинформированность о комиссии              123
Неправомерное списание комиссии              17
Неправильное списание комиссии               15
Неожиданная комиссия                         15
Комиссия за обслуживание                     14
                                           ... 
Неясный лимит начисления процентов            1
Отсутствие информации в личном кабинете       1
Неполное закрытие счета                       1
Отсутствие уведомления о комиссиях            1
Неудобство оформления повторного запроса      1
Name: is, Length: 1112, dtype: int64

In [359]:
q['ref'].explode().value_counts()

Возврат комиссии                               363
Возврат средств                                 39
Refund commission                               24
Закрытие карты                                  15
Возврат списанной суммы                         14
                                              ... 
Разъяснение причины блокировки оплаты            1
Устранение проблемы с SMS‑уведомлениями          1
Предоставление правовой информации               1
Возврат необоснованной комиссии                  1
Предоставление расчёта начисления процентов      1
Name: ref, Length: 897, dtype: int64

### text2vec

In [420]:
def expand_display(col_width=None, max_cols=None, total_width=None):
    pd.set_option('display.max_colwidth', col_width if col_width is not None else 1000)
    pd.set_option('display.max_columns', max_cols if max_cols is not None else None)
    pd.set_option('display.width', total_width if total_width is not None else 1000)

# Пример использования
expand_display(col_width=200, max_cols=50, total_width=2000)

In [466]:
q_is = q.explode('is').dropna(subset=['is']).reset_index(drop=True)
r_is = q.explode('ref').dropna(subset=['ref']).reset_index(drop=True)

In [467]:
q_is.shape, r_is.shape

((1607, 6), (1568, 6))

In [468]:
sp=list()
for i, row in tqdm.tqdm(q_is.iterrows()):
    payload['input'] = row['is']
    response = requests.post(url, headers=headers, data=json.dumps(payload))
    a = json.loads(response.content)['data'][0]['embedding']
    sp.append(a)
    # break

1607it [00:47, 34.18it/s]


In [469]:
sp2=list()
for i, row in tqdm.tqdm(r_is.iterrows()):
    payload['input'] = row['ref']
    response = requests.post(url, headers=headers, data=json.dumps(payload))
    a = json.loads(response.content)['data'][0]['embedding']
    sp2.append(a)
    # break

1568it [00:45, 34.25it/s]


In [470]:
km = KMeans(n_clusters=8)
km.fit(sp)
pr_sp = km.predict(sp)

In [471]:
km = KMeans(n_clusters=30)
km.fit(sp2)
pr_sp2 = km.predict(sp2)

In [472]:
q_is['cls'] = pr_sp
q_is.cls.value_counts()

2    247
3    224
5    215
7    214
4    208
6    202
1    155
0    142
Name: cls, dtype: int64

In [473]:
r_is['cls'] = pr_sp2
r_is.cls.value_counts()

1     437
8      61
5      61
17     60
9      59
14     56
18     49
6      47
15     47
16     47
23     44
13     44
22     42
29     41
26     41
12     40
7      39
25     35
28     34
21     31
10     29
2      27
11     27
0      26
4      26
27     25
20     25
24     25
19     24
3      19
Name: cls, dtype: int64

In [479]:
# q_is[q_is.cls==4][['is_d', 'is']].sample(15)
r_is[r_is.cls==23][['ref', 'ref_d']]#.sample(15)

,ref,ref_d
79,Бесплатное обслуживание карты,"[Вернуть клиенту удержанную сумму комиссии в размере 99 руб., Перевести карту клиента на тариф без ежемесячной платы за обслуживание.]"
118,Сохранение карты,"[Вернуть клиенту удержанную сумму комиссии за обслуживание карты., Обеспечить клиенту возможность продолжать пользоваться зарплатной картой без дополнительных комиссий.]"
150,Выдача договора открытия счета и карты,[Необходимо подготовить и отправить клиенту договор открытия счета и карты в удобном для него виде (электронно или в бумажной форме).]
170,Перевыпуск карты,"[Выпустить клиенту новую карту и заблокировать текущую, чтобы предотвратить дальнейшее мошенничество., Остановить повторяющиеся списания, инициировать возврат уже списанных средств и открыть спор ..."
174,Продление срока действия карты,"[Предоставить клиенту новый срок действия текущей карты без необходимости её перевыпуска., Выдать клиенту официальное письмо, в котором указаны причины отказа в продлении срока действия карты и сс..."
203,Проверка статуса старой карты,"[Предоставить клиенту подробное объяснение, почему карта работает нестабильно в банкоматах., Уточнить, была ли ранее выданная карта клиента заблокирована и какие действия были предприняты., Выясни..."
204,Проверка наличия второй карты,"[Предоставить клиенту подробное объяснение, почему карта работает нестабильно в банкоматах., Уточнить, была ли ранее выданная карта клиента заблокирована и какие действия были предприняты., Выясни..."
232,Проверка и привязка карты,"[Вернуть клиенту удержанную комиссию в размере 800 рублей за снятие наличных в стороннем банкомате., Уточнить, к какой карте относится обращение (физическое лицо или ИП), проверить её статус и, пр..."
254,Ускорение выпуска карты,"[Срочно завершить процесс изготовления социальной карты и подготовить её к выдаче., Предложить клиенту возможность получения карты через курьерскую доставку или оформить доверенность для получения..."
337,Помощь в привязке карты через MirAccept,"[Сообщить клиенту телефон отделения банка в городе Клинцы, Брянская область., Показать клиенту шаги по обновлению приложения и корректной привязке карты к мессенджерам., Осуществить замену устарев..."


In [486]:
', '.join(r_is['ref'].unique().tolist())

'Возврат средств за билет, Организация встречи, Запрос выписки, Оспаривание операции, Инициировать чарджбек, Возврат средств, Возврат комиссии, Разъяснение условий обслуживания, Инструкция по переводу, Возврат списанных средств, Проверка записей звонков, Корректировка информации о задолженности, Компенсация за причинённый ущерб, Разрешение проблемы с зачислением пенсии, Объяснение политики комиссии, Возврат комиссии EUR, Возврат комиссии USD, Письменный ответ, Информирование по телефону, Отмена комиссии, Отказ от требования депозита, Отмена возврата комиссии, Зачисление пенсии на карту, Урегулирование причины возврата, Проверка выписок, Перерасчет и возврат, Проверка телефонных записей, Перерасчет и списание задолженности, Уведомление клиента о результатах, Предоставление расшифровки списаний, Расследование причины отказа, Проверка фактов открытия счета, Закрытие нежелательного счета, Разбор причины списаний, Перерасчёт и выплата недостающих процентов, Сохранение старых условий обслужи

In [463]:
r_is[r_is.cls==15][['ref_d']].sample(25).sum()[0]

'Вернуть клиенту списанную сумму 4200\u202f₽ после проверки транзакций.Вернуть клиенту полностью списанную сумму пенсионных накоплений.Провести проверку причин списания средств с карты клиента и вернуть ошибочно списанные 306,00\u202f₽ (в сумме 612,00\u202f₽).Вернуть клиенту списанную сумму 2999,00\u202fруб за обслуживание карты.Вернуть клиенту списанные суммы в размере 3000\u202f₽ + 3000\u202f₽ (итого 6000\u202f₽) на карту 2****0858.Вернуть клиенту списанные 999\u202f₽ за обслуживание.Вернуть клиенту списанные 99\u202f₽ за услугу SMS‑информирования, которую он не подключал.Вернуть клиенту сумму 982\u202f₽, ошибочно списанную в счет ареста, и снять блокировку средств.Вернуть клиенту полностью сумму в размере 9\u202f680\u202f₽, списанную без законных оснований.Вернуть клиенту списанную сумму 4200\u202f₽ на карту, с которой была произведена оплата (карта Уралсиб).Отменить и вернуть списанные средства за платное обслуживание.Вернуть клиенту удержанную сумму в размере 1400\u202f₽ в качеств

In [398]:
response.content

b'{"error":{"message":"[{\'type\': \'list_type\', \'loc\': (\'body\', \'function-wrap[__log_extra_fields__()]\', \'input\', \'list[int]\'), \'msg\': \'Input should be a valid list\', \'input\': nan}, {\'type\': \'list_type\', \'loc\': (\'body\', \'function-wrap[__log_extra_fields__()]\', \'input\', \'list[list[int]]\'), \'msg\': \'Input should be a valid list\', \'input\': nan}, {\'type\': \'string_type\', \'loc\': (\'body\', \'function-wrap[__log_extra_fields__()]\', \'input\', \'str\'), \'msg\': \'Input should be a valid string\', \'input\': nan}, {\'type\': \'list_type\', \'loc\': (\'body\', \'function-wrap[__log_extra_fields__()]\', \'input\', \'list[str]\'), \'msg\': \'Input should be a valid list\', \'input\': nan}, {\'type\': \'missing\', \'loc\': (\'body\', \'function-wrap[__log_extra_fields__()]\', \'messages\'), \'msg\': \'Field required\', \'input\': {\'input\': nan, \'model\': \'BAAI/bge-m3\', \'encoding_format\': \'float\'}}]","type":"Bad Request","param":null,"code":400}}

In [271]:
q.requested_actions.apply(lambda x: extract_values(x))

0    [Refund processing, Провести возврат средств з...
1    [Запрос выписки, Клиент просит предоставить вы...
2    [Принятие оспаривания, Официально признать тра...
3    [Возврат комиссии, Вернуть клиенту удержанную ...
4    [Инструкция по переводу, Предоставить клиенту ...
5    [Возврат комиссии, Вернуть клиенту удержанную ...
6    [Возврат средств, Вернуть клиенту удержанную с...
7    [Возврат списанных средств, Вернуть клиенту су...
8    [Возврат комиссии, Вернуть клиенту ошибочно сп...
9    [Разрешение проблемы с зачислением пенсии, Нео...
Name: requested_actions, dtype: object

In [222]:
b = [{'type': 'Refund request for non‑refundable ticket',
  'description': 'Клиент хочет вернуть деньги за односторонний билет, который по правилам считается невозвратным.'},
 {'type': 'VIP service handling',
  'description': 'Клиент требует особого обслуживания как VIP и просит организовать встречу с представителем компании.'}]

In [226]:
extract_keys(b)

['type', 'description', 'type', 'description']

In [210]:
json.loads(sp[3][:])

{'issues': [{'type': 'Неинформированность о комиссии',
   'description': 'Клиент не был проинформирован о наличии комиссии за обслуживание карты в договоре и в отделении банка, хотя списание 999\u202f₽ произошло 23.10.2025.'},
  {'type': 'Неавторизованное списание',
   'description': 'С 2022 года списаний за обслуживание не было, а теперь без согласия клиента списана сумма 999\u202f₽, что считается ошибкой.'}],
 'requested_actions': [{'type': 'Возврат комиссии',
   'description': 'Вернуть клиенту удержанную сумму 999\u202f₽ и закрыть спор по данному списанию.'},
  {'type': 'Предоставление информации о комиссии',
   'description': 'Предоставить клиенту полную информацию о тарифах и условиях обслуживания карты, а также объяснить, почему карта перестала быть зарплатной.'}]}

In [207]:
['issues']

TypeError: 'function' object is not subscriptable

In [131]:
# a = json.loads(response.content)['choices']#[0]['message']['content']

In [200]:
df_small.desc[2]

'Добрый день. Далее со слов клиента: "От: Ривина Михаила Сергеевича По карте:ПС МИР, маска ***1102 Контактный телефон:+7-951-644-63-20  Заявление об оспаривании операции  Прошу оспорить операцию и вернуть сумму 299 рублей, списанную с моей карты безоснавательно.  Данные операции: · Дата/время: 06.10.2024 14:18 МСК · Сумма: 299 руб. · Получатель: GAZPROMBONUS. · Основание в выписке: «подписка на сервис».  Причина оспаривания. Подписку на данный сервис я не подключал и согласия на списание не давал.  Ключевое доказательство: При обращении к GAZPROMBONUS мне отказали в возврате, предоставив в качестве «доказательства» подписку по номеру телефона +7-916-041-63-57.  Заявляю, что... 1. Указанный номер +7-916-041-63-57 мне НЕ принадлежит. 2. Мой реальный номер, привязанный к банковской карте — +7-951-644-63-20. 3. На мой реальный номер в сервисе GAZPROMBONUS активных подписок нет.  Таким образом, операция является несанкционированной, а «доказательства» со стороны сервиса — не релевантны.  На

None


In [ ]:
Сформируй ответ в формате JSON, описывающий проблему обращения клиента.



В ответе допускаются только два корневых ключа:

1. **issues** – массив объектов, каждый объект — отдельная проблема клиента (может отсутствовать, если проблем нет);

2. **requested_actions** – массив объектов, каждый объект — действие, которое клиент просит выполнить (может отсутствовать, если запросов нет).



Для каждого объекта в массиве указывай минимум два поля:

- **type** – краткое название/категория проблемы или запроса;

- **description** – развернутое описание.



**Требования к формату**

- JSON‑документ должен быть валидным (правильные кавычки, запятые, скобки).

- Порядок ключей: `issues` → `requested_actions` (если оба присутствуют).

- Если один из массивов пустой, его можно опустить полностью или оставить как пустой массив `[]`.

- Не добавляй лишних полей и не используй вложенные структуры, отличные от описанных выше.



**Пример правильного ответа**



```json

{

  "issues": [

    {

      "type": "Неинформированность о комиссии",

      "description": "Клиент не был предупреждён о комиссии за снятие наличных в сторонних банкоматах."

    },

    {

      "type": "Закрытый счёт",

      "description": "Счёт дебетовой карты закрыт, поэтому работодатель не может перечислить зарплату."

    }

  ],

  "requested_actions": [

    {

      "type": "Возврат комиссии",

      "description": "Вернуть клиенту удержанную комиссию в размере 3200 ₽."

    },

    {

      "type": "Открытие нового счёта",

      "description": "Создать новый дебетовый счёт и предоставить реквизиты работодателю."

    }

  ]

}

Проблема:

1

In [78]:
from collections import defaultdict
import tqdm
out = defaultdict(list)
for i, row in tqdm.tqdm(df.iterrows()):
    payload['input'] = row['desc']
    response = requests.post(url, headers=headers, data=json.dumps(payload))
    vec = json.loads(response.content)['data'][0]['embedding']
    out['num'].append(row['num'])
    out['vec'].append(vec)
    # break

2905it [01:38, 29.50it/s]


In [80]:
df.shape

(2905, 7)

In [87]:
from sklearn.cluster import KMeans

In [88]:
km = KMeans(n_clusters=15)

In [89]:
km.fit(out['vec'])

KMeans(n_clusters=15)

array([ 2,  3, 14, ...,  4,  1,  4], dtype=int32)

In [91]:
out['cls'] = km.predict(out['vec'])

In [94]:
pd.DataFrame([out['num'], out['cls']])/T

NameError: name 'T' is not defined

In [98]:
df = df.reset_index(drop=True)

In [99]:
df['cls'] = km.predict(out['vec'])

In [102]:
df[['stage3','desc','cls']].to_excel('s.xlsx', index=False)

In [103]:
df.cls.value_counts()

12    441
4     271
14    258
9     254
7     230
1     229
6     227
10    201
3     189
0     158
13    134
2      96
11     91
5      65
8      61
Name: cls, dtype: int64

In [3]:
import pandas as pd

In [18]:
import pathlib

In [33]:
import pathlib
import codecs

# Если в проекте уже используется chardet
# pip install chardet
import chardet

def detect_csv_encoding(file_path: str | pathlib.Path, sample_size: int = 100_000) -> str:
    """
    Возвращает строку с названием кодировки, которую удалось определить
    для CSV‑файла. Делает это, читая небольшой фрагмент (по умолчанию
    до 100 KB) и используя библиотеку chardet.
    """
    file_path = pathlib.Path(file_path)

    # Читаем часть файла (чтобы не перегружать память)
    with file_path.open("rb") as f:
        raw_data = f.read(sample_size)

    # Определяем кодировку
    result = chardet.detect(raw_data)

    # result = {"encoding": "utf-8", "confidence": 0.99, "language": ""}
    encoding = result["encoding"]
    confidence = result["confidence"]

    if not encoding:
        raise ValueError("Не удалось определить кодировку.")

    # При низкой уверенности можно вернуть fallback‑кодировку
    if confidence < 0.8:
        print(f"Низкая уверенность ({confidence:.2%}) – "
              f"может потребоваться ручная проверка.")
    return encoding


# Пример использования
if __name__ == "__main__":
    csv_path = "./output - 2025-10-29T194929.261.csv"

    try:
        enc = detect_csv_encoding(csv_path)
        print(f"Определённая кодировка: {enc}")

        # Проверка – открываем файл с найденной кодировкой
        with open(csv_path, encoding=enc) as f:
            # читаем первые несколько строк, чтобы убедиться, что всё ок
            for _ in range(5):
                print(f.readline().rstrip())
    except Exception as e:
        print(f"Ошибка: {e}")

Определённая кодировка: UTF-16
Номер	Дата регистрации	Раб. дней в работе	Плановая дата начала	Фактическая дата начала	ФИО клиента	Кол-во активных обращений	 LTV	Дата акт. LTV	Тип	Подтип	Претензия по продукту	Наименование продукта	Статус	Канал	Приоритет рассмотрения	Связано с претензией	Связано со сбоем	Исполнитель ДПР	Отв. дирекция	Профильное подразделение	Дата ответа проф. подразделения	Дата отправки запроса	Зарегистрировано	Принявшее подразделение	Родительская ОШС	Дочерняя ОШС	Основной ДУЛ	ID клиента в IR	Номер в программе УралсибБонус	Дата принятия решения	План. дата окончания	Факт. дата окончания	Дата предоставления ответа	Описание претензии	ФИО сотрудника, отмеченного положительным отзывом	Тип заявления	ФИО виновного сотрудника	Сумма требований	Сумма фактических выплат	Сумма фактических выплат УРАЛСИБ БОНУС	Примечания	Оценка претензии	Офис, на который жалуется клиент	Тип клиента	Валюта выплаты	Валюта требований	Номер устройства	Адрес устройства	Дата и время операции	Сумма операции

In [31]:
!pip install chardet

Looking in indexes: https://nexus/repository/pypi-proxy/simple


In [32]:
import chardet

/tmp/ipykernel_629/2515919784.py:1: DtypeWarning: Columns (12,17,47,59,66) have mixed types. Specify dtype option on import or set low_memory=False.
  df = pd.read_csv('output - 2025-10-29T194929.261.csv', sep="\t", encoding='UTF-16')


In [52]:
df = pd.read_csv('output - 2025-10-29T194929.261.csv', sep="\t", encoding='UTF-16')
cols = ['Номер',
'Дата регистрации',
'Тип', 'Подтип',
       'Претензия по продукту', 'Канал', 'Описание претензии']
df = df[cols]
df.columns = ['num', 'date', 'stage2', 'stage3', 'stage1', 'channel', 'desc']

df = df[df.stage1.isin(['Дебетовая карта']) & df.stage2.isin(['Обслуживание по карте'])]

In [56]:
df.reset_index(drop=True).stage3.unique()

array(['Не доволен тарифами / тарифной политикой / условиями договора',
       'Платеж / перевод не исполнен / исполнен несвоевременно',
       'Отмена ежегодной комиссии за РКО',
       'Несогласие с образованием технической / дебиторской задолженности',
       'Несвоевременное зачисление денежных средств', 'Mir Pass',
       'Отмена ежемесячной комиссии за РКО',
       'Отмена комиссии за обслуживание неактивного картсчета',
       'Не согласен с начисленными процентами',
       'Проблема с перевыпуском карты',
       'Разблокировка средств по операции',
       'Проблема с 3-D secure / не приходят Смс / Push',
       'Отказ/ошибка в проведении операций по карте',
       'Проблема с активацией карты',
       'Не согласен с конвертацией / курсовой разницей',
       'Клиент не согласен с удержанием НДФЛ',
       'Отмена разовой комиссии за РКО'], dtype=object)

In [57]:
df3

NameError: name 'df3' is not defined

In [7]:
df.columns = ['type', 'subtype', 'product']

In [14]:
df[df['product'].isin(['Дебетовая карта'])]

,type,subtype,product
5,Арест/блокировка картсчета,Не согласен с блокировкой счета,Дебетовая карта
16,Закрытие карты,Не произошло закрытие карты и счета по заявлению,Дебетовая карта
17,Арест/блокировка картсчета,Не согласен с блокировкой счета,Дебетовая карта
20,Закрытие карты,Вопрос по закрытию,Дебетовая карта
29,Закрытие карты,Вопрос по закрытию,Дебетовая карта
...,...,...,...
54061,Обслуживание по карте,Не доволен тарифами / тарифной политикой / усл...,Дебетовая карта
54069,Обслуживание по карте,Не согласен с начисленными процентами,Дебетовая карта
54076,Обслуживание по карте,Не согласен с начисленными процентами,Дебетовая карта
54082,Обслуживание по карте,Отмена ежемесячной комиссии за РКО,Дебетовая карта


**Контекст**  
Для анализа обращения — необходимо отделять саму причину жалобы (то, что спровоцировало её возникновение) от её последствий. Причина — это конкретный факт, ошибочная операция или некорректное действие системы (например, неверный расчёт комиссии, ошибка в тарифах, неправильное информирование и т.п.). Последствия (списание средств, блокировка карты, ухудшение сервиса) являются лишь результатом этой причины и не подлежат классификации. При категоризации обращения мы фиксируем именно ту проблему, из‑за которой клиент получил недовольство, т. е. классифицируем её с точки зрения клиента, а не с точки зрения внутренних процессов банка.
При формировании ответа используйте именно эти названия категорий в поле **issues**.  

**Категории проблем (issues)**  

| **Category** | **Sub‑category** | **Description (кратко)** |
|--------------|------------------|---------------------------|
| **1. Комиссии и сборы** | 1.1 Неожиданная / скрытая комиссия | Платёж списан без предварительного информирования (комиссия за обслуживание, за снятие, за пополнение и т.п.). |
| | 1.2 Комиссия за неактивный/неиспользуемый счёт | Списание за отсутствие активности (дебетовый, бизнес‑счёт, карта «прибыль», премиум‑пакет). |
| | 1.3 Комиссия за бизнес‑залы / премиум‑услуги | Плата за доступ к бизнес‑залам, премиум‑картам без согласия. |
| | 1.4 Комиссия за операции RKO / пополнение | Списание за расчётно‑кассовое обслуживание, за пополнение через партнёров, за ввод/вывод наличных. |
| | 1.5 Комиссия за международные/online‑платежи | Плата за переводы в рамках SBP, SWIFT, онлайн‑покупки, за превышение лимитов. |
| | 1.6 Неправильный расчёт комиссии | Ошибочное начисление суммы комиссии (дублирование, слишком высокая ставка). |
| **2. Неавторизованные/незаконные списания** | 2.1 Неавторизованные списания по счету (RUB/EUR/USD) | Списание средств без согласия владельца (в том числе за страховку, подписки, услуги). |
| | 2.2 Неавторизованные операции в бизнес‑залах | Неодобренный доступ к бизнес‑залам и последующее списание. |
| | 2.3 Неавторизованное открытие/закрытие счёта | Открытие нового счета/карты без согласия клиента. |
| | 2.4 Неавторизованные комиссии за услуги | Плата за услуги, о которых клиент не знал (SMS‑информирование, 3‑DS, кэш‑бэк). |
| **3. Проблемы с картой** | 3.1 Блокировка/заморозка карты | Карта заблокирована банком без объяснения (по ФЗ‑161, подозрению в fraude и т.п.). |
| | 3.2 Ошибки при выпуске/перевыпуске карты | Не получена карта, задержка выдачи, неправильный тип карты, карта с ошибочным номером. |
| | 3.3 Проблемы с PIN / 3‑DS / верификацией | Неправильный номер телефона, отсутствие возможности установить/сменить 3‑DS, ошибка PIN. |
| | 3.4 Неподдерживаемый/некомплатный тип карты | Карта не подходит для онлайн‑покупок, QR‑платежей, MirAccept, Яндекс Go. |
| | 3.5 Потеря/кража карты и последствия | Утеря, кража, блокировка, невозможность восстановить средства. |
| **4. Проблемы со счётом и балансом** | 4.1 Неотображение/недоступность средств | Баланс не показывается, средства «заморожены», отсутствие баланса в приложении. |
| | 4.2 Ошибки начисления процентов/пени | Неправильный расчёт процентов, отсутствие начисления, избыточные пени. |
| | 4.3 Неправильное списание/зачисление средств | Ошибочный перевод, неверный получатель, двойное/тройное списание, возврат на закрытый счёт. |
| | 4.4 Долги и задолженности, неясные данные | Неожиданная задолженность, непонятные обороты, неучтённые операции. |
| | 4.5 Закрытие/открытие счёта без уведомления | Закрытие счета/карты без сообщения, невозможность открыть новый счёт онлайн. |
| **5. Переводы и платежи** | 5.1 Ошибки при переводе (SBP, QR, SWIFT) | Неправильный получатель, отказ в переводе, задержка зачисления, двойные переводы. |
| | 5.2 Отказы в оплате (POS, онлайн, QR) | Отклонённые покупки, превышение лимитов, неверный MCC‑код, отказ в сервисе доставки. |
| | 5.3 Необходимость подтверждения/верификации | Требование кода, 3‑DS, SMS‑подтверждение, невозможность подтвердить операцию. |
| | 5.4 Проблемы с вводом/выводом наличных | Ошибки при пополнении в банкомате, ограничение приёма банкнот, отсутствие наличных. |
| **6. Премиальные/бонусные программы** | 6.1 Кэш‑бэк и бонусы не начислены | Пропущенные кэш‑бэки, недополученные бонусы, снижение кэш‑бэка без уведомления. |
| | 6.2 Ограничения и закрытие премиум‑пакетов | Отказ в премиум‑обслуживании, закрытие пакета без объяснения, блокировка привилегий. |
| **7. Пенсии и социальные выплаты** | 7.1 Не поступление пенсии/пособий | Задержка выплаты, отсутствие выплат, неверный номер счёта, потеря выплат. |
| | 7.2 Ошибки в расчёте/перечислении социальных средств | Неверный оборот, отсутствие выплат по соц‑программам. |
| **8. Информационная поддержка и коммуникация** | 8.1 Отсутствие/противоречивая информация | Нет ответа в чате, разные ответы от колл‑центра и процессинга, неверные консультации. |
| | 8.2 Низкое качество обслуживания | Непрофессиональное общение, задержки в рассмотрении, отсутствие менеджера. |
| | 8.3 Недостаточная информированность о тарифах/изменениях | Нет уведомления о новых тарифах, скрытые изменения условий, отсутствие описания комиссии. |
| | 8.4 Нарушение конфиденциальности / утечки данных | Неавторизованная передача персональных данных, публикация данных клиента. |
| **9. Документы и справки** | 9.1 Отсутствие/запрос справок и выписок | Не предоставили выписку для суда, нет справки о доходах, отсутствие договоров. |
| | 9.2 Ошибки в договорах и согласиях | Нет договора с фондом, отсутствие согласия на списание, нелегальное изменение условий. |
| **10. Технические и системные проблемы** | 10.1 Проблемы мобильного/веб‑приложения | Ошибки в личном кабинете, отсутствие SMS‑уведомлений, 3‑DS не работает, приложение «зависает». |
| | 10.2 Сбои в работе автоплатежей и сервисов | Не работают автоплатежи, ошибка при оплате картой, сбой QR‑платежа. |
| | 10.3 Ограничения доступа к сервисам | Нет доступа к онлайн‑банкингу, невозможность привязать карту к сервису, отсутствие 3‑DS. |
| **11. Инфраструктурные ограничения** | 11.1 Нет отделения/банкомата в регионе | Отсутствие ближайшего отделения, партнёрских банкоматов, ограниченный доступ к наличным. |
| | 11.2 Ограничения лимитов (снятие, пополнение) | Лимиты в банкоматах, ограничения на пополнение, ограничение на перевод средств. |
| **12. Юридические / нормативные вопросы** | 12.1 Применение 115‑ФЗ, 161‑ФЗ и др. | Блокировка по законодательным требованиям без объяснения. |
| | 12.2 Споры с кредитными бюро (БКИ) | Ошибки в кредитной истории, необоснованные запросы. |
| **13. Маркетинг и нежелательные контакты** | 13.1 Навязчивая реклама / звонки | Рекламные звонки, рассылка предложений, невозможность отказаться от новых условий. |
| **14. Прочие/разные** | 14.1 Ошибки реквизитов (БИК, ИНН, номер телефона) | Неправильный БИК при оплате, неверный ИНН в чеке, устаревший номер телефона. |
| | 14.2 Ошибки в классификации MCC‑кода | Неправильный MCC‑код, приводящий к неверному начислению комиссии. |
| | 14.3 Прочие ситуации без отдельной категории | Утечка данных, нарушение прав потребителей, отсутствие возможности изменить тариф самостоятельно и т.п. |

При классификации учитывай следующее:
1. Если клиент жалуется, что ему **неправильно сообщили о тарифе/комиссии**, выбирай категорию **8. Информационная поддержка и коммуникация** → **8.1 Отсутствие/противоречивая информация** (или 8.3 Недостаточная информированность о тарифах/изменениях, если более точно).

**Запросы клиента (requested_actions)**  
При обработке обращения необходимо выделять каждый конкретный запрос клиента – это чётко сформулированное требование выполнить определённое действие (например, «вернуть комиссию», «открыть новый счёт», «разблокировать карту», «выдать выписку» и т.п.) – и оформлять его в массив requested_actions, где указываются три обязательных поля: category (короткое название действия, используем только указанные в инструкции названия, например «Возврат комиссии», «Разблокировка карты», «Выдача выписки»), sub‑category (при необходимости уточняющая подкатегория) и description (развёрнутое описание запроса с указанием всех деталей, приведённых клиентом). Каждый найденный запрос оформляется отдельным объектом в массиве; если запросов несколько, они перечисляются последовательно. Отклонения от предложенного формата недопустимы – только точные названия категорий и полное описание гарантируют корректную обработку обращения.

При формировании ответа используйте именно эти названия категорий в поле **requested_actions**.  

**Категории запросов клиента  (requested_actions)**  

| **Category** | **Sub‑category** | **Краткое описание** |
|--------------|------------------|----------------------|
| **1. Возврат / возмещение средств** | 1.1 Возврат средств за товар/услугу (билет, страхование, комиссия, штраф) | Требование вернуть уплаченную сумму полностью или частично. |
| | 1.2 Возврат ошибочно списанных комиссий (EUR, USD, RKO, SMS‑информирование, бизнес‑залы) | Списание комиссии без согласия/основания – запрос возврата. |
| | 1.3 Возврат переплаты / двойного списания | Сумма была списана два‑три раза – нужно вернуть излишки. |
| | 1.4 Возврат удержанных средств (пенсия, зарплата, кэшбэк, бонусы) | Необходимо вернуть удержанные средства, которые должны были быть зачислены. |
| | 1.5 Экспресс‑возврат (через СБП, на карту, наличными) | Быстрый возврат в случае невозможности завершить перевод. |
| | 1.6 Возврат в виде компенсации (моральный ущерб, за просрочку, за неудобства) | Компенсация финансовых потерь и/или морального вреда. |
| **2. Корректировка/перерасчёт операций** | 2.1 Перерасчёт процентов / пени | Пересчёт начисленных процентов, исправление ошибок расчёта. |
| | 2.2 Перерасчёт комиссии / сброс комиссии | Перерасчёт комиссии, её отмена или уменьшение. |
| | 2.3 Корректировка баланса (дебетовая/кредитная карта) | Исправление ошибочного остатка, списаний или зачислений. |
| | 2.4 Списание/аннулирование задолженности (долги, пени, овердрафт) | Полное снятие задолженности или её частичное списание. |
| | 2.5 Корректировка кредитной/дебетовой истории | Исправление записей о просроченных платежах, штрафах. |
| | 2.6 Перераспределение/переназначение платежей | Перевод средств между своими счетами/картами. |
| **3. Отмена / прекращение списаний и услуг** | 3.1 Отмена текущей комиссии / автоматических списаний | Запрет дальнейшего списания комиссии, RKO, подписок и т.п. |
| | 3.2 Отмена/аннулирование услуги (премиум‑пакет, овердрафт, страховка) | Прекращение действия платных сервисов. |
| | 3.3 Отмена перевода/транзакции | Остановить или отменить уже инициированный платёж. |
| | 3.4 Прекращение рекламных/сборных звонков | Блокировать нежелательные телефонные обращения. |
| **4. Управление счётом и картой** | 4.1 Закрытие/анулирование счёта/карты | Закрыть нежелательный счёт/карту, включая «полное» закрытие. |
| | 4.2 Перевыпуск/выдача новой карты (бесплатно, с новым сроком) | Выпуск/замена карты и/или изменение тарифов обслуживания. |
| | 4.3 Блокировка/разблокировка карты и онлайн‑банка | Временная блокировка, разблокировка, снятие ограничений. |
| | 4.4 Снятие ограничений на дистанционные покупки, снятие лимитов | Увеличение лимитов, снятие блокировок по онлайн‑транзакциям. |
| | 4.5 Обновление данных (телефон, адрес, 3‑DS, номер телефона) | Смена контактных данных, изменение настроек 3‑DS. |
| | 4.6 Привязка/отвязка карты к сервисам (MirAccept, SBP, QR) | Подключение/отключение карты в внешних приложениях. |
| **5. Информационные запросы и разъяснения** | 5.1 Разъяснение политики комиссии/тарифов/лимитов | Объяснение, почему начислена комиссия, как формируются тарифы. |
| | 5.2 Запрос выписок, расшифровок операций, справок | Предоставление выписок, детализации движений, справок о доходах. |
| | 5.3 Информирование о статусе обращения/запроса | Уведомление о текущем статусе, результатах расследования. |
| | 5.4 Предоставление нормативных актов, договора, условий | Выдать политику банка, договоры, правила пользования. |
| | 5.5 Письменный/официальный ответ (на адрес, в email) | Составление официального письма‑ответа клиенту. |
| | 5.6 Объяснение причин отказа, блокировки, отклонения транзакции | Детальное обоснование отрицательного решения. |
| **6. Расследование и проверка** | 6.1 Проверка и расследование спорных списаний/транзакций | Выяснение причины нежелательных списаний, их возврат. |
| | 6.2 Проверка записей звонков, чатов, аудиозаписей | Анализ клиентского общения для подтверждения фактов. |
| | 6.3 Внутреннее расследование действий сотрудников | Выяснение возможных нарушений со стороны персонала. |
| | 6.4 Проверка фактов открытия/закрытия счёта, статуса карт | Подтверждение/отклонение заявлений о создании/закрытии. |
| | 6.5 Проверка расчётов по процентам, кэшбэку, обороту | Аудит расчётов начислений и проверка корректности. |
| **7. Защита данных и конфиденциальность** | 7.1 Удаление/анонимизация персональных данных (из отзывов, системы) | Удалить персональные сведения по требованию клиента. |
| | 7.2 Прекращение неавторизованного доступа к данным | Блокировать дальнейшее использование личных данных. |
| | 7.3 Обеспечение соответствия обработке персональных данных (GDPR/ФЗ‑152) | Гарантировать законность обработки клиентской информации. |
| **8. Техническая и сервисная поддержка** | 8.1 Улучшение работы чата/горячей линии, обучение операторов | Повышение качества общения, предоставление скриптов. |
| | 8.2 Настройка/восстановление SMS‑уведомлений, 3‑DS, кода доступа | Включение/корректировка уведомлений и механизмов верификации. |
| | 8.3 Исправление ошибок в мобильном/веб‑приложении (баланс, операции) | Технические правки в личном кабинете. |
| | 8.4 Ускорение обработки заявок, сокращение сроков (экспресс‑расмотрение) | Приоритетное решение обращений. |
| | 8.5 Внедрение новых функций (дистанционная установка кода, автоплатеж) | Добавление/активация сервисов по запросу клиента. |
| **9. Прочие запросы** | 9.1 Организация встречи/консультации в отделении | Запрос личного визита сотрудника/встречи. |
| | 9.2 Предоставление контактных данных филиала, номера телефона | Информация о точках обслуживания. |
| | 9.3 Перевод средств на другие банковские карты/счета (включая SWIFT) | Выполнение переводов, в том числе международных. |
| | 9.4 Открытие новых счетов/карт (по доверенности, рублевый счёт) | Создание новых продуктов в рамках текущего профиля. |
| | 9.5 Установка банкомата/камеры в офисе | Техническое обслуживание инфраструктуры. |
| | 9.6 Прекращение рекламных рассылок, спама | Блокировка маркетинговых контактов. |
| | 9.7 Согласование условий реструктуризации, рассрочки, каникул по кредиту | Переговоры о изменении условий кредитования. |
| | 9.8 Оформление претензий, жалоб, эскалаций, передача в надзорные органы | Составление и подача официальных жалоб. |
| | 9.9 Предоставление информации о лимите, кэшбэке, бонусах | Разъяснение правил начисления и использования бонусов. |
| | 9.10 Доступ к архивным данным, восстановление закрытых/архивированных счетов | Выдача исторических выписок, восстановление старых записей. |

### Инструкция для модели

1. **Проанализируйте текст обращения** (в переменной `{{text}}`).  
2. **Определите одну или несколько проблем** и сопоставьте каждую из них с одной из категорий из таблицы выше.  
3. **Сформируйте массив `issues`** – каждый элемент содержит поля `category`, `sub-category` (выбранное из списка категорий и подкатегорий) и `description` (кратко, но полностью отражающее суть проблемы).  
4. **Если в тексте присутствует явный запрос клиента**, сформируйте массив `requested_actions` с соответствующими элементами  – каждый элемент содержит поля `category`, `sub-category` (выбранное из списка категорий и подкатегорий) и `description` (кратко, но полностью отражающее суть запроса).  
5. **Сгенерируйте валидный JSON**:  
   * Корневые ключи – только `issues` и `requested_actions` (в указанном порядке).  
   * Если один из массивов пустой, его можно опустить либо оставить как `[]`.  
   * Не добавляйте никаких дополнительных полей или вложенных структур.  

На основе вышеизложенных инструкций разметь следующий запрос:
{{text}}